In [101]:
from WindPy import w
import numpy as np
import pandas as pd
import datetime
import os
import json
from tqdm import tqdm

w.start()
w.isconnected()

True

In [102]:
with open("INDEX_START.json", "r", encoding="utf-8") as f:
    INDEX_START = json.load(f)

In [103]:
df = pd.read_csv("A股指数.csv")
a_share_dict = dict(zip(df["Ticker"], df["Name"]))

df = pd.read_csv("海外指数.csv")
other_market_dict = dict(zip(df["Ticker"], df["Name"]))

print(len(a_share_dict), len(other_market_dict))

333 57


In [104]:
len(INDEX_START)

188

# Load Local Database

In [113]:
INDEX_DATA = {}

for ticker in INDEX_START:
    path = f"Data/{ticker}.csv"
    
    if os.path.isfile(path): 
        data = pd.read_csv(path, index_col = 0, parse_dates = True)
        data.index = pd.to_datetime(data.index, format="%Y-%m-%d", errors='coerce')
        data.index = data.index.date
        INDEX_DATA[ticker] = data.copy(deep = True)
        
len(INDEX_DATA)

186

In [106]:
print(INDEX_DATA['000300.SH'].index[-1])

2025-05-14


# Update Existing Tickers

In [108]:
today = "2025-05-15"
today_date = datetime.datetime.strptime(today, "%Y-%m-%d").date()
today_date

datetime.date(2025, 5, 15)

In [109]:
# for ticker in tqdm(INDEX_DATA):
#     data = INDEX_DATA[ticker]

#     idx = -1
#     last = date.index[idx]
#     while not (type(last) is datetime.date):
#         idx -= 1
#         last = date.index[idx]

#     start = (last + pd.Timedelta(days = 1)).strftime("%Y-%m-%d")
    
#     new_data = w.wsd(ticker, 'pe_ttm,dividendyield2,close', start, today, '', usedf = True)[1]

#     if len(new_data) == 1 and new_data.index[0] == ticker:
#         new_data.index = [today_date]
    
#     if new_data.columns[0] == 'OUTMESSAGE':
#         print(ticker)
#         continue
    
#     INDEX_DATA[ticker] = pd.concat([data, new_data])
#     INDEX_DATA[ticker].dropna(inplace = True)

In [114]:
for ticker in tqdm(INDEX_DATA):
    data = INDEX_DATA[ticker]

    if data.index[-1] == today_date:
        continue
    
    start = data.index[-1].strftime("%Y-%m-%d")

    new_data = w.wsd(ticker, 'pe_ttm,dividendyield2,close', start, today, '', usedf = True)[1]

    data = pd.concat([data, new_data]).groupby(level = 0).last()
    data.index = pd.to_datetime(data.index, format="%Y-%m-%d", errors='coerce')
    data.index = data.index.date

    data = data[~data.index.duplicated(keep = "last")]

    # if len(new_data) == 1 and new_data.index[0] == ticker:
    #     new_data.index = [today_date]
    
    if new_data.columns[0] == 'OUTMESSAGE':
        print(ticker)
        continue

    INDEX_DATA[ticker] = data.copy(deep = True)
    INDEX_DATA[ticker].dropna(inplace = True)

100%|████████████████████████████████████████████████████████████████████████████████| 186/186 [00:02<00:00, 79.67it/s]


In [122]:
for ticker in tqdm(INDEX_DATA):
    data = INDEX_DATA[ticker]

    data = data[~data.index.duplicated(keep = "last")]

    INDEX_DATA[ticker] = data.copy(deep = True)

100%|██████████████████████████████████████████████████████████████████████████████| 186/186 [00:00<00:00, 3172.37it/s]


In [111]:
data.tail()

,PE_TTM,DIVIDENDYIELD2,CLOSE
2025-05-09,15.4853,2.8856,2733.49
2025-05-12,15.4745,2.8896,2742.08
2025-05-13,15.8105,2.8702,2772.14
2025-05-14,15.6817,2.8880,2763.29
2025-05-15,15.5607,2.9254,2738.96


# Download New Tickers

In [11]:
for ticker, start in tqdm(INDEX_START.items()):
    if ticker in INDEX_DATA:
        continue
        
    assert ticker not in INDEX_DATA
    
    data = w.wsd(ticker, 'pe_ttm,dividendyield2,close', start, today, '', usedf = True)[1]
    
    junk = data['PE_TTM']

    if data.isna().sum().sum() != 0:
        print(ticker)
    
    data.dropna(inplace = True)
    
    INDEX_DATA[ticker] = data.copy(deep = True)

 99%|███████████████████████████████████████▌| 187/189 [00:00<00:00, 270.33it/s]

SP6CSSUP.SPI


100%|█████████████████████████████████████████| 189/189 [00:02<00:00, 84.13it/s]


In [125]:
for ticker, data in INDEX_DATA.items():
    path = os.path.join("Data", f"{ticker}.csv")
    data.to_csv(path, index = True, encoding = "utf-8")

In [12]:
len(INDEX_DATA)

131

In [116]:
INDEX_DATA['000913.SH'].tail()

,PE_TTM,DIVIDENDYIELD2,CLOSE
2025-05-13,28.240000,1.8197,7891.3630
2025-05-14,28.418400,1.8083,7959.5245
2025-05-15,28.409901,1.8086,7968.7475
2025-05-14,28.418400,1.8083,7959.5245
2025-05-15,28.409901,1.8086,7968.7475


In [123]:
temp = INDEX_DATA['000913.SH']


In [124]:
temp.tail()

,PE_TTM,DIVIDENDYIELD2,CLOSE
2025-05-09,28.068100,1.9990,7848.4305
2025-05-12,27.993900,1.8358,7818.1952
2025-05-13,28.240000,1.8197,7891.3630
2025-05-14,28.418400,1.8083,7959.5245
2025-05-15,28.409901,1.8086,7968.7475
